# PubMed Guideline Search Notebook

This notebook is converted from the original `PubMed_guideline_search.py` script. It is designed for a **stroke guideline review** database-search pathway.

Main workflow:

1. Define PubMed search configuration.
2. Build the final Boolean query.
3. Search PubMed using NCBI E-utilities.
4. Fetch all records in XML format.
5. Parse PMID, title, journal, year, DOI, and publication types.
6. De-duplicate records by DOI and normalized title.
7. Export raw and deduplicated CSV files.
8. Print PRISMA identification counts.

> **Important:** Run this notebook on a machine with internet access. Some cloud/sandbox environments may block outbound network requests.

## 0. Optional: install dependencies

The core script only uses Python standard library modules.

`pandas` is only used at the end for convenient CSV preview. If your environment does not have `pandas`, you can install it or skip the preview cell.

In [1]:
# Optional, only if pandas is missing in your environment
# !pip install pandas

## 1. Import libraries

In [2]:
import time
import csv
import json
import re
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen, Request

## 2. Configuration

Edit this cell before running the search.

NCBI asks users to provide a real contact email. An API key is optional, but it allows faster requests.

In [ ]:
# ----------------------------- CONFIG ---------------------------------
EMAIL = "your_email@example.com"      # REQUIRED: replace with your real email
API_KEY = ""                   # OPTIONAL: paste an NCBI API key for faster requests
TOOL = "stroke_guideline_review"

# Output folder. Use Path(".") for the current notebook directory.
OUTPUT_DIR = Path(".")

# Your PubMed search query. Edit freely. This mirrors the Methods search strategy.
QUERY = (
    '(stroke[Title/Abstract] OR cerebrovascular[Title/Abstract] OR '
    '"intracerebral haemorrhage"[Title/Abstract] OR "intracerebral hemorrhage"[Title/Abstract] OR '
    '"subarachnoid haemorrhage"[Title/Abstract] OR "subarachnoid hemorrhage"[Title/Abstract] OR TIA[Title/Abstract]) '
    'AND (guideline[Publication Type] OR "practice guideline"[Publication Type] OR '
    'guideline[Title] OR guidelines[Title] OR consensus[Title] OR recommendation*[Title] OR standard[Title]) '
    'AND (rehabilitation[Title/Abstract] OR assessment[Title/Abstract] OR "outcome measure*"[Title/Abstract] OR '
    'gait[Title/Abstract] OR balance[Title/Abstract] OR mobility[Title/Abstract] OR motor[Title/Abstract])'
)

DATE_FROM = "2014"     # Publication year lower bound. Set "" to disable.
DATE_TO = "2026"       # Publication year upper bound. Set "" to disable.
LANGUAGES = []         # Example: ["english", "chinese", "japanese"]. [] = no language filter.
# ----------------------------------------------------------------------

BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

## 3. Validate configuration

This replaces the original script's `sys.exit()` behavior. In a notebook, raising a clear error is easier to handle.

In [5]:
def validate_config():
    """Check whether required user settings have been edited."""
    if EMAIL == "you@example.com" or not EMAIL.strip():
        raise ValueError("Please replace EMAIL with your real contact email before running the PubMed search.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Uncomment after editing EMAIL
# validate_config()

## 4. API helper functions

These functions prepare NCBI request parameters, send requests, and respect NCBI rate limits.

In [6]:
def _params(extra):
    """Attach required NCBI metadata and optional API key to request parameters."""
    p = {"tool": TOOL, "email": EMAIL}
    if API_KEY:
        p["api_key"] = API_KEY
    p.update(extra)
    return p


def _get(url):
    """Send a GET request and return decoded text."""
    req = Request(url, headers={"User-Agent": f"{TOOL} ({EMAIL})"})
    with urlopen(req, timeout=60) as r:
        return r.read().decode("utf-8", "replace")


def _rate_sleep():
    """Respect NCBI rate limits: 3 requests/sec without API key, 10 requests/sec with API key."""
    time.sleep(0.12 if API_KEY else 0.34)

## 5. Build the final PubMed query

In [7]:
def build_query():
    """Combine the Boolean query with optional date and language filters."""
    q = QUERY

    if DATE_FROM and DATE_TO:
        q += f' AND ({DATE_FROM}:{DATE_TO}[dp])'

    if LANGUAGES:
        langs = " OR ".join(f'{lang}[lang]' for lang in LANGUAGES)
        q += f' AND ({langs})'

    return q


final_query = build_query()
print(final_query)

(stroke[Title/Abstract] OR cerebrovascular[Title/Abstract] OR "intracerebral haemorrhage"[Title/Abstract] OR "intracerebral hemorrhage"[Title/Abstract] OR "subarachnoid haemorrhage"[Title/Abstract] OR "subarachnoid hemorrhage"[Title/Abstract] OR TIA[Title/Abstract]) AND (guideline[Publication Type] OR "practice guideline"[Publication Type] OR guideline[Title] OR guidelines[Title] OR consensus[Title] OR recommendation*[Title] OR standard[Title]) AND (rehabilitation[Title/Abstract] OR assessment[Title/Abstract] OR "outcome measure*"[Title/Abstract] OR gait[Title/Abstract] OR balance[Title/Abstract] OR mobility[Title/Abstract] OR motor[Title/Abstract]) AND (2014:2026[dp])


## 6. Search PubMed with ESearch

This step only retrieves the total count and PubMed history-server information. It does **not** download all article records yet.

In [8]:
def esearch(query):
    """Run NCBI ESearch and return count, WebEnv, and query_key."""
    url = f"{BASE}/esearch.fcgi?" + urlencode(
        _params({
            "db": "pubmed",
            "term": query,
            "retmax": 0,
            "usehistory": "y",
            "retmode": "json",
        })
    )

    data = json.loads(_get(url))
    _rate_sleep()

    res = data["esearchresult"]
    return int(res["count"]), res["webenv"], res["querykey"]

In [9]:
# Run this cell after editing EMAIL in the config cell.
validate_config()

count, webenv, query_key = esearch(final_query)
print(f"[ESearch] records identified from PubMed = {count}")

[ESearch] records identified from PubMed = 932


## 7. Parse PubMed XML records

This parser extracts the minimal fields needed for screening and de-duplication.

In [10]:
def parse_pubmed_xml(xml):
    """Parse PubMed XML and extract minimal bibliographic fields."""
    records = []

    for article in re.findall(r"<PubmedArticle>.*?</PubmedArticle>", xml, re.S):
        def grab(pattern):
            match = re.search(pattern, article, re.S)
            if not match:
                return ""
            return re.sub(r"<[^>]+>", "", match.group(1)).strip()

        pmid = grab(r"<PMID[^>]*>(.*?)</PMID>")
        title = grab(r"<ArticleTitle>(.*?)</ArticleTitle>")
        journal = grab(r"<Title>(.*?)</Title>")
        year = grab(r"<PubDate>.*?<Year>(.*?)</Year>")

        doi = ""
        doi_match = re.search(r'<ELocationID EIdType="doi"[^>]*>(.*?)</ELocationID>', article, re.S)
        if doi_match:
            doi = re.sub(r"<[^>]+>", "", doi_match.group(1)).strip().lower()

        publication_types = "; ".join(
            re.findall(r"<PublicationType[^>]*>(.*?)</PublicationType>", article)
        )

        records.append({
            "pmid": pmid,
            "title": title,
            "journal": journal,
            "year": year,
            "doi": doi,
            "pub_types": publication_types,
        })

    return records

## 8. Fetch all PubMed records

Records are fetched in batches through PubMed's history server.

In [11]:
def efetch_all(count, webenv, query_key, batch=200):
    """Fetch PubMed records in batches and parse each XML response."""
    records = []

    for start in range(0, count, batch):
        url = f"{BASE}/efetch.fcgi?" + urlencode(
            _params({
                "db": "pubmed",
                "WebEnv": webenv,
                "query_key": query_key,
                "retstart": start,
                "retmax": batch,
                "retmode": "xml",
            })
        )

        xml = _get(url)
        _rate_sleep()

        records.extend(parse_pubmed_xml(xml))
        print(f"   fetched {min(start + batch, count)}/{count}")

    return records

In [12]:
if count == 0:
    records = []
    print("No records found. Consider loosening the query and rerunning the search.")
else:
    records = efetch_all(count, webenv, query_key, batch=200)
    print(f"Total parsed records = {len(records)}")

   fetched 200/932
   fetched 400/932
   fetched 600/932
   fetched 800/932
   fetched 932/932
Total parsed records = 917


## 9. De-duplicate records

The logic follows the original script:

1. First remove duplicates by DOI.
2. Then remove remaining duplicates by normalized article title.

In [13]:
def norm_title(title):
    """Normalize title text for title-based de-duplication."""
    return re.sub(r"[^a-z0-9]+", " ", title.lower()).strip()


def dedup(records):
    """De-duplicate records by DOI first, then by normalized title."""
    seen_doi = set()
    seen_title = set()
    deduped = []
    duplicate_count = 0

    for record in records:
        doi_key = record["doi"] or None
        title_key = norm_title(record["title"])

        is_duplicate = (
            (doi_key and doi_key in seen_doi) or
            (title_key and title_key in seen_title)
        )

        if is_duplicate:
            duplicate_count += 1
            continue

        if doi_key:
            seen_doi.add(doi_key)
        if title_key:
            seen_title.add(title_key)

        deduped.append(record)

    return deduped, duplicate_count

In [14]:
deduped_records, duplicate_count = dedup(records)

print(f"Duplicate records removed = {duplicate_count}")
print(f"Records after de-duplication = {len(deduped_records)}")

Duplicate records removed = 33
Records after de-duplication = 884


## 10. Export raw and de-duplicated CSV files

In [15]:
def write_csv(path, rows):
    """Write rows to a UTF-8-SIG CSV file for Excel compatibility."""
    path = Path(path)

    if not rows:
        path.write_text("", encoding="utf-8-sig")
        return

    with path.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

In [16]:
raw_csv_path = OUTPUT_DIR / "pubmed_records_raw.csv"
dedup_csv_path = OUTPUT_DIR / "pubmed_records_dedup.csv"

write_csv(raw_csv_path, records)
write_csv(dedup_csv_path, deduped_records)

print(f"Raw records saved to: {raw_csv_path.resolve()}")
print(f"Deduplicated records saved to: {dedup_csv_path.resolve()}")

Raw records saved to: D:\Pycharm\JupyterProject1\pubmed_records_raw.csv
Deduplicated records saved to: D:\Pycharm\JupyterProject1\pubmed_records_dedup.csv


## 11. Print PRISMA identification counts

Paste these values into the `PubMed_search_log` sheet of your search documentation template.

In [17]:
print("================  PRISMA identification: databases  ================")
print(f"Records identified from PubMed .................... {count}")
print(f"Duplicate records removed ........................ {duplicate_count}")
print(f"Records after de-duplication ..................... {len(deduped_records)}")
print("Screen 'pubmed_records_dedup.csv' by title/abstract.")
print("Record how many you exclude and how many full texts you seek.")
print("====================================================================")

================  PRISMA identification: databases  ================
Records identified from PubMed .................... 932
Duplicate records removed ........................ 33
Records after de-duplication ..................... 884
Screen 'pubmed_records_dedup.csv' by title/abstract.
Record how many you exclude and how many full texts you seek.


## 12. Optional: preview the de-duplicated records

In [18]:
try:
    import pandas as pd
    from IPython.display import display

    df_dedup = pd.read_csv(dedup_csv_path)
    print(f"Previewing {min(10, len(df_dedup))} of {len(df_dedup)} deduplicated records")
    display(df_dedup.head(10))
except ImportError:
    print("pandas is not installed. You can still open the exported CSV file directly.")
except FileNotFoundError:
    print("CSV file not found. Run the export cell first.")

Previewing 10 of 884 deduplicated records


,pmid,title,journal,year,doi,pub_types
0,42129593,Diagnosis and Management of Patent Foramen Ova...,The Medical journal of Australia,2026,10.5694/mja2.70199,"<PublicationType UI=""D016428"">Journal Article;..."
1,42120775,Asian expert consensus on high-quality hyperte...,Hypertension research : official journal of th...,2026,10.1038/s41440-026-02644-2,"<PublicationType UI=""D016428"">Journal Article;..."
2,42095756,2025 update to European Stroke Organisation (E...,European stroke journal,2026,10.1093/esj/aakag004,"<PublicationType UI=""D016428"">Journal Article;..."
3,42095754,"European Stroke Organisation (ESO), European A...",European stroke journal,2026,10.1093/esj/aakag043,"<PublicationType UI=""D016428"">Journal Article;..."
4,42070673,Factors associated with discharge to skilled n...,Journal of stroke and cerebrovascular diseases...,2026,10.1016/j.jstrokecerebrovasdis.2026.108653,"<PublicationType UI=""D016428"">Journal Article;..."
5,42057665,Effect of intensive versus standard blood pres...,Annals of medicine,2026,10.1080/07853890.2026.2662627,"<PublicationType UI=""D016428"">Journal Article;..."
6,42038467,Development and validation of standard-criteri...,PeerJ,2026,10.7717/peerj.21050,"<PublicationType UI=""D016428"">Journal Article;..."
7,42026607,Recommendations for rehabilitation of post-str...,BMC health services research,2026,10.1186/s12913-026-14594-z,"<PublicationType UI=""D016428"">Journal Article;..."
8,41990337,"Lipoprotein(a), High-Sensitivity C-Reactive Pr...",European journal of preventive cardiology,2026,10.1093/eurjpc/zwag221,"<PublicationType UI=""D016428"">Journal Article"
9,41969626,American Epilepsy Society Clinical Practice Gu...,Epilepsy currents,2026,10.1177/15357597261433266,"<PublicationType UI=""D016428"">Journal Article"


## Notes for manuscript / methods writing

PubMed may under-index clinical practice guidelines because many authoritative guidelines are hosted only on organization websites. Therefore, this database-search pathway should be reported as a **supplementary database pathway**, not as a replacement for targeted searches of guideline-issuing organizations.